# 📈 Scikit-learn Regression Pipeline ONNX with SnoozeML

This notebook demonstrates:
1. **Train** a scikit-learn `Pipeline` (`StandardScaler` + `Ridge` regression) on the standard Diabetes dataset.
2. **Export** the multi-step preprocessing and regressor pipeline to ONNX.
3. **Upload / Deploy** via **HTTP multipart upload** directly to SnoozeML (no volume mounts or daemon restarts required).
4. **Cold Start vs. Warm Execution**: Measure cold start load time vs. ultra-low latency in-memory inference, evaluating $R^2$ and MSE.

In [1]:
from pathlib import Path
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error

from snoozeml import SnoozeClient
from snoozeml.converters import export_sklearn

## 1. Load Data & Fit Pipeline

In [2]:
# Load Diabetes regression dataset (10 features)
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an end-to-end Pipeline with scaling and ridge regression
pipe = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_train, y_train)
print(f"Training R² score: {pipe.score(X_train, y_train):.4f}")

Training R² score: 0.5276


## 2. Export & Upload Pipeline Directly to SnoozeML via HTTP

SnoozeML compiles the scaler normalization and ridge linear regression into a fused ONNX graph. Upload it directly over HTTP with custom runtime configuration.

In [3]:
# Export pipeline locally
model_path = Path("./diabetes_pipeline.onnx")
export_sklearn(pipe, sample_input=X_train, input_name="input", output_path=model_path)
print(f"Pipeline exported locally to {model_path} ({model_path.stat().st_size} bytes)")

# Connect to SnoozeML daemon
# If authentication is enabled, supply your API key or JWT token:
# client = SnoozeClient(endpoint="http://localhost:8080", api_key="snooze_admin_secret_key")
client = SnoozeClient(endpoint="http://localhost:8080")

# Upload directly via HTTP with custom runtime configuration (no volume mounts required!)
model_info = client.upload_model(
    file=model_path,
    model_id="diabetes_pipeline",
    description="Scikit-learn Pipeline (StandardScaler + Ridge Regression) on Diabetes",
    pinned=False,               # Keep unpinned to allow scale-to-zero memory reclamation
    max_batch_size=16,
    max_batch_delay_ms=10,
    idle_timeout_secs=300,
)
print(f"Uploaded model: {model_info.id} | Initial State: {model_info.state.value} | Pinned: {model_info.pinned}")

# Note: Runtime model refresh can also be triggered at any time:
# client.refresh_models()

Pipeline exported locally to diabetes_pipeline.onnx (651 bytes)
Uploaded model: diabetes_pipeline | Initial State: idle | Pinned: False


## 3. Compare Cold Start vs. Warm In-Memory Inference

Demonstrating SnoozeML's dynamic loading on cold start and warm in-memory inference.

In [4]:
# Ensure model starts in idle state (scale-to-zero)
client.models.evict("diabetes_pipeline")
print(f"State before 1st request: {client.models.get('diabetes_pipeline').state.value}")

# --- 1st Request: Cold Start (eager weight loading into memory + forward pass) ---
resp_cold = client.infer(model_id="diabetes_pipeline", inputs={"input": X_test})
print(f"❄️  Cold Start Latency: {resp_cold.duration_ms:.2f} ms (weights loaded on-demand)")
print(f"State after 1st request:  {client.models.get('diabetes_pipeline').state.value}")

# --- 2nd Request: Warm Execution (model already resident in memory) ---
resp_warm = client.infer(model_id="diabetes_pipeline", inputs={"input": X_test})
print(f"🔥 Warm Latency:        {resp_warm.duration_ms:.2f} ms (in-memory forward pass)")

speedup = resp_cold.duration_ms / max(resp_warm.duration_ms, 0.01)
print(f"⚡ Speedup:             {speedup:.1f}x faster warm execution")

# Evaluate regression metrics
preds = resp_warm.get_numpy().flatten()
print(f"SnoozeML Test R²:    {r2_score(y_test, preds):.4f}")
print(f"SnoozeML Test MSE:   {mean_squared_error(y_test, preds):.2f}")
print(f"Predicted targets:   {preds[:5].round(1)}")
print(f"Ground truth:        {y_test[:5].round(1)}")

State before 1st request: idle
❄️  Cold Start Latency: 31.49 ms (weights loaded on-demand)
State after 1st request:  ready
🔥 Warm Latency:        0.81 ms (in-memory forward pass)
⚡ Speedup:             39.0x faster warm execution
SnoozeML Test R²:    0.4541
SnoozeML Test MSE:   2892.01
Predicted targets:   [139.9 180.  135.7 292.1 123.2]
Ground truth:        [219.  70. 202. 230. 111.]
